# Pipeline 154 — original 8-run results, direct native path

Notebook-only replacement. Original notebook/outputs: `archive/Pipeline_154.pre-native-history.ipynb`. The monitoring agent's cell is retained unchanged. No helper execution class, wrapper, worker or backend edit. Use the existing `.quantum` kernel. Inspection is read-only. All 24 own saved bindings are completed: verify and show original results/winners, never retrain. Native missing-work calls are separate, explicit and require exact own constructor/model records; they do not inherit Pipeline 155 settings.


In [ ]:
from pathlib import Path
from copy import deepcopy
import hashlib
import json
import math
import sys
import tempfile

PIPELINE_ID = 154
TARGET_RUNS = 8
sys.dont_write_bytecode = True
FRAMEWORK_ROOT = Path('/Users/pitergarcia/DataScience/Semester4/GA-Work/hybrid_variable_framework/Dynamic_Routing_Eval_Framework')
sys.path.insert(0, str(FRAMEWORK_ROOT))
from daqr.pipeline.pipeline import Pipeline
pipeline = Pipeline(PIPELINE_ID)
assert pipeline.target_runs == TARGET_RUNS
workflow_document = pipeline.read_workflow()
assert workflow_document['plan']['runs'] == TARGET_RUNS
expected_sets = {allocator + ':' + replay + ':' + scale
    for allocator in workflow_document['plan']['allocators']
    for replay in ('T', 'Tb') for scale in ('1', '1.5', '2')}
assert len(expected_sets) == 24
assert set(workflow_document['savedBindings']) == expected_sets
cache_path = Path(tempfile.gettempdir()) / 'tae-workflow-status' / f'Pipeline_{PIPELINE_ID}.json'
if cache_path.is_file():
    cached_document = json.loads(cache_path.read_bytes())['document']
    if cached_document.get('savedBindings') != workflow_document['savedBindings']:
        print('Local workflow cache differs; using this pipeline own live Git references')
print(f'Pipeline {PIPELINE_ID}: {TARGET_RUNS} runs; 24 own saved binding references')


## Read-only inspection — verify each pinned Git result binding once

Configuration artifacts below preserve `recovered_config`, `savedScientificSettings`, per-threat settings references and original runner references as recorded. Missing historical model kwargs/allocator constructor records are NOT replaced by current defaults. They are unnecessary for displaying completed results; they remain required for any future missing-work execution.


In [ ]:
from googleapiclient.errors import HttpError
import pickle
saved_artifact_errors = (AssertionError, OSError, ValueError, KeyError, TypeError,
                         EOFError, ImportError, HttpError, pickle.UnpicklingError)
blocked_restore = []
def report_optional(label, operation, *args, **kwargs):
    try:
        return operation(*args, **kwargs)
    except Exception as error:
        print(f'{label}: {type(error).__name__}: {error}', flush=True)

from daqr.config.gd_backup_manager import GoogleDriveBackupManager
sys.path.insert(0, str(FRAMEWORK_ROOT / 'notebooks'))
from cohort_run_publication import SharedCohortStore
cohort_store = SharedCohortStore()
pipeline_git_key = f'pipeline-{PIPELINE_ID}'
git_results = workflow_document['gitSavedResults']
assert git_results['repository'] == cohort_store.repository
assert set(git_results['bindings']) == expected_sets
verified_result_tags = set()
def pipeline_drive_service():
    service = globals().get('drive_service')
    if service is None:
        service = GoogleDriveBackupManager.pipeline_creation_drive_service()
        globals()['drive_service'] = service
    return service
saved_byte_cache = globals().get('saved_byte_cache', {})
bindings = {}
configuration_artifacts = {}
missing_work = []
for set_id, reference in workflow_document['savedBindings'].items():
    try:
        key = (reference['provider_id'], reference['content_sha256'], reference['byte_length'])
        if key not in saved_byte_cache:
            pinned_result = git_results['bindings'][set_id]
            allocator_id, replay_id, scale_id = set_id.split(':')
            cohort_id = f'S{scale_id}{replay_id}'
            assert pinned_result['cohort'] == cohort_id
            assert pinned_result['path'] == f'saved-results/{allocator_id}.json'
            assert (pinned_result['size'], pinned_result['sha256']) == (key[2], key[1])
            expected_tag = f"versions/{pipeline_git_key}/{git_results['execution']}/{cohort_id}/{pinned_result['commit']}"
            assert pinned_result['tag'] == expected_tag
            if expected_tag not in verified_result_tags:
                if cohort_store.ref('tags/' + expected_tag) != pinned_result['commit']:
                    raise ValueError('Saved-result tag does not match its pinned commit: ' + set_id)
                verified_result_tags.add(expected_tag)
            saved_byte_cache[key] = cohort_store.read(pinned_result['commit'], pinned_result['path'], key[2], key[1])
        raw = saved_byte_cache[key]
        assert len(raw) == reference['byte_length'], set_id
        assert hashlib.sha256(raw).hexdigest() == reference['content_sha256'], set_id
        binding = json.loads(raw)
        own_config = binding['recovered_config']
        allocator_name, replay, scale = set_id.split(':')
        assert own_config['runs'] == TARGET_RUNS, set_id
        assert own_config['allocator'] == allocator_name, set_id
        assert own_config['base_capacity'] == (replay == 'Tb'), set_id
        assert float(own_config['scale']) == float(scale), set_id
        assert set(own_config['test_scenarios']) == set(workflow_document['plan']['threats'])
        assert own_config['models'] == workflow_document['plan']['models']
        assert set(binding['env_experiments']) == set(own_config['test_scenarios'])
        set_missing = []
        for threat, experiments in binding['env_experiments'].items():
            assert set(experiments) <= {str(position) for position in range(1, TARGET_RUNS + 1)}
            for position in range(1, TARGET_RUNS + 1):
                experiment = experiments.get(str(position))
                if experiment is None:
                    set_missing.append((set_id, threat, position))
                    continue
                assert set(experiment['results']) == set(own_config['models']), (set_id, threat, position)
                if experiment.get('winner') not in (None, 'NA', 'N/A'):
                    assert experiment['winner'] in own_config['models'], (set_id, threat, position)
                for model, result in experiment['results'].items():
                    if result.get('error') or not math.isfinite(float(result['final_reward'])):
                        raise ValueError('Invalid saved result; preserve it: ' + set_id + '/' + threat + '/' + model)
                    assert result['frames_count'] == own_config['base_frames'] + (position-1)*own_config['frame_step']
        configuration_artifacts[set_id] = dict(
            sourceRef=deepcopy(reference), recovered_config=deepcopy(own_config),
            savedScientificSettings=deepcopy(binding['savedScientificSettings']),
            unrecordedScientificFields=deepcopy(binding.get('unrecordedScientificFields', [])),
            originalSettingsRefs=deepcopy(binding.get('continuation', {}).get('originalSettingsRefs', {})),
            runnerReferences=deepcopy(binding.get('continuation', {}).get('runnerReferences', {})))
        bindings[set_id] = binding
        missing_work.extend(set_missing)
    except saved_artifact_errors as error:
        blocked_restore.append(set_id)
        print(f"{set_id}: saved binding rejected: {type(error).__name__}: {error}", flush=True)
        continue
print('Verified own bindings:', len(bindings), '; saved target outcomes:',
      sum(len(experiment['results']) for binding in bindings.values()
          for experiments in binding['env_experiments'].values() for experiment in experiments.values()))
print('Missing experiment positions:', len(missing_work), '; no training was invoked')


## Original monitoring cell — retained unchanged

This is the original monitoring cell, not an execution engine. It publishes status when the user runs it. It is not executed during read-only notebook validation.


In [ ]:
from copy import deepcopy
from types import SimpleNamespace
import re
from daqr.config import tae_workflow

monitor = tae_workflow.current_tracker(
    SimpleNamespace(pipeline_storage=pipeline.storage_binding))
if monitor is None:
    raise ValueError('Monitoring requires this pipeline storage binding')
monitor.ensure_source()
monitor_plan = deepcopy(pipeline.read_workflow()['plan'])
assert monitor_plan['runs'] == pipeline.target_runs
if 'plan' in monitor.document and monitor.document['plan'] != monitor_plan:
    raise ValueError('Recorded monitoring plan differs; preserve state and reconcile first')
monitor.document.setdefault('plan', monitor_plan)
for allocator_name in monitor_plan['allocators']:
    for cohort_name in monitor_plan['cohorts']:
        cohort_match = re.fullmatch(r'S([0-9.]+)(Tb|T)', cohort_name)
        if cohort_match is None:
            raise ValueError('Unsupported recorded cohort: ' + cohort_name)
        planned = SimpleNamespace(
            pipeline_storage=pipeline.storage_binding, allocator=allocator_name,
            scale=float(cohort_match[1]), base_capacity=cohort_match[2] == 'Tb',
            runs=monitor_plan['runs'], models=monitor_plan['models'],
            test_scenarios=monitor_plan['threats'], overwrite=False)
        for threat in monitor_plan['threats']:
            planned.attack_type = threat
            for position in range(1, monitor_plan['runs'] + 1):
                existing = (monitor.document.get('allocators', {})
                    .get(tae_workflow.safe_key(allocator_name), {}).get('cohorts', {})
                    .get(cohort_name, {}).get('threats', {})
                    .get(tae_workflow.safe_key(threat.lower()), {}).get('experiments', {}))
                if str(position) not in existing:
                    monitor.transition(planned, position, 'none')
                experiment = (monitor.document['allocators'][tae_workflow.safe_key(allocator_name)]
                    ['cohorts'][cohort_name]['threats'][tae_workflow.safe_key(threat.lower())]
                    ['experiments'][str(position)])
                for model_name in monitor_plan['models']:
                    experiment.setdefault('models', {}).setdefault(
                        tae_workflow.safe_key(model_name), dict(state='none'))
monitor.pending = True
monitor.write_live_status()


## Completed-work branch — show original saved results and winner

These are the recorded model metrics and recorded winner, not recomputed winners or newly trained results. The completed branch does not instantiate an evaluator, runner, allocator or model. It does not need guessed historical model kwargs to display saved results. Each completed position is explicitly skipped.


In [ ]:
def display_saved_pipeline_results():
    from types import SimpleNamespace
    from daqr.evaluation.multi_run_evaluator import MultiRunEvaluator

    for set_id, binding in bindings.items():
        own_config = binding['recovered_config']
        print('\n' + '='*90 + '\nPIPELINE', PIPELINE_ID, set_id)
        for threat in own_config['test_scenarios']:
            for position, experiment in sorted(binding['env_experiments'][threat].items(), key=lambda pair: int(pair[0])):
                frames = own_config['base_frames'] + (int(position)-1)*own_config['frame_step']
                capacity = own_config['base_frames'] if own_config['base_capacity'] else frames
                scenario = own_config['test_scenarios']
                details = scenario.get(threat) if isinstance(scenario, dict) else None
                details = details if isinstance(details, dict) else {}
                display_context = SimpleNamespace(frames_count=frames, configs=SimpleNamespace(
                    allocator=own_config['allocator'], environment=details.get('env_type', own_config.get('env_type', 'not recorded')),
                    attack_strategy=threat, attack_rate=details.get('attack_rate', own_config.get('attack_rate', 'not recorded'))))
                saved = dict(experiment, winner=experiment.get('winner') or 'NA')
                experiment_id = int(position)
                print(f"[{own_config['allocator']}] {threat}: ⏩ SKIPPING EXPERIMENT {experiment_id}: ALREADY COMPLETED AND STORED")
                MultiRunEvaluator.display_run_results(display_context, experiment_id,
                    {experiment_id: saved}, capacity * own_config['scale'])
    if not bindings:
        print('No verified saved results to display; see the saved-binding messages.')


display_saved_pipeline_results()


## Original full-result contract recovery — data-only, no execution

Optional historical inspection only; native resume does not use this cell. Set `FULL_RESULT_SET_IDS = ['Default:T:1']` to inspect a selected subset. The inert reader checks the original evaluator and runner payloads without restoring an evaluator or executing any algorithm.


In [ ]:
import importlib.util

full_result_bindings = globals().get('full_result_bindings', {})
FULL_RESULT_SET_IDS = globals().get(
    'FULL_RESULT_SET_IDS', [])
original_payload_paths = globals().get('original_payload_paths', {})
if FULL_RESULT_SET_IDS:
    reader_path = Path('/Users/pitergarcia/.local/share/dsci602/data-preparation/run-six-continuation-20261108/extend_set.py')
    inspector_path = reader_path.parent.parent / 'pipeline6-materialized-five-run-20261026/inspect_full_source54.py'
    assert hashlib.sha256(reader_path.read_bytes()).hexdigest() == '4e76d7252e25ae043c4c212aeec07dd91d546e73a5f13d94d29422a0d2d4974e'
    assert hashlib.sha256(inspector_path.read_bytes()).hexdigest() == '2683f9bfa1672fb083ae354e81344d2064ad9708f9391d215e53ab1ebf094c1f'
    reader_spec = importlib.util.spec_from_file_location('_existing_inert_result_reader', reader_path)
    result_reader = importlib.util.module_from_spec(reader_spec)
    reader_spec.loader.exec_module(result_reader)
    for set_id in FULL_RESULT_SET_IDS:
        full_result_bindings.pop(set_id, None)
        try:
            chain = [bindings[set_id]]
            seen_prefixes = set()
            while chain[-1].get('continuation', {}).get('originalPrefix'):
                reference = chain[-1]['continuation']['originalPrefix']
                key = (reference['provider_id'], reference['content_sha256'], reference['byte_length'])
                if key in seen_prefixes or len(chain) >= TARGET_RUNS:
                    raise ValueError('Cyclic or unbounded original prefix chain: ' + set_id)
                seen_prefixes.add(key)
                if key not in saved_byte_cache:
                    saved_byte_cache[key] = pipeline_drive_service().files().get_media(
                        fileId=reference['provider_id'], supportsAllDrives=True).execute()
                raw = saved_byte_cache[key]
                assert len(raw) == reference['byte_length']
                assert hashlib.sha256(raw).hexdigest() == reference['content_sha256']
                previous = json.loads(raw)
                assert previous['recovered_config']['runs'] < chain[-1]['recovered_config']['runs']
                assert previous['recovered_config']['allocator'] == bindings[set_id]['recovered_config']['allocator']
                assert previous['recovered_config']['scale'] == bindings[set_id]['recovered_config']['scale']
                assert previous['recovered_config']['base_capacity'] == bindings[set_id]['recovered_config']['base_capacity']
                chain.append(previous)
            recovered_results = {}
            with tempfile.TemporaryDirectory(prefix=f'pipeline-{PIPELINE_ID}-original-payloads-') as directory:
                payload_directory = Path(directory)
                for source_binding in reversed(chain):
                    source_runs = source_binding['recovered_config']['runs']
                    reference = source_binding['sourceRef']
                    if reference['original_filename'].endswith('.pkl'):
                        payload_references = [('full-evaluator', reference)]
                    else:
                        runner_references = source_binding.get('continuation', {}).get('runnerReferences', {})
                        if set(runner_references) != set(source_binding['recovered_config']['test_scenarios']):
                            raise ValueError('Original full runner references missing: ' + set_id)
                        payload_references = list(runner_references.items())
                    for threat, reference in payload_references:
                        key = (reference['provider_id'], reference['content_sha256'], reference['byte_length'])
                        payload_path = original_payload_paths.get(key)
                        if payload_path is None:
                            payload_path = payload_directory / reference['original_filename']
                            raw = pipeline_drive_service().files().get_media(
                                fileId=reference['provider_id'], supportsAllDrives=True).execute()
                            assert len(raw) == reference['byte_length']
                            assert hashlib.sha256(raw).hexdigest() == reference['content_sha256']
                            payload_path.write_bytes(raw)
                            del raw
                        else:
                            payload_path = Path(payload_path)
                            assert payload_path.stat().st_size == reference['byte_length']
                            assert hashlib.sha256(payload_path.read_bytes()).hexdigest() == reference['content_sha256']
                        payload = result_reader.read_checkpoint(payload_path)
                        if threat == 'full-evaluator':
                            assert payload['runs_id'] == source_runs
                            recovered_results = {
                                scenario: {int(position): deepcopy(experiment)
                                    for position, experiment in experiments.items()}
                                for scenario, experiments in payload['env_experiments'].items()}
                        else:
                            assert payload['id'] == source_runs and payload.get('is_complete')
                            assert set(payload['results']) == set(source_binding['recovered_config']['models'])
                            recovered_results.setdefault(threat, {})[source_runs] = dict(
                                results=deepcopy(payload['results']), winner=payload.get('winner'),
                                exp_id=source_runs)
                        del payload
                    result_reader.reject_inert_payload(recovered_results)
                    for scenario, experiments in source_binding['env_experiments'].items():
                        for position, projected in experiments.items():
                            original_experiment = recovered_results[scenario][int(position)]
                            assert set(original_experiment['results']) == set(projected['results'])
                            for model, reduced in projected['results'].items():
                                original_result = original_experiment['results'][model]
                                assert original_result['final_reward'] == reduced['final_reward']
                                assert original_result['frames_count'] == reduced['frames_count']
                                if (not isinstance(original_result.get('model_results'), dict)
                                        or 'reward_list' not in original_result['model_results']):
                                    raise ValueError('Original archive lacks native reward history: '
                                        + set_id + '/' + scenario + '/' + str(position) + '/' + model)
                own_config = bindings[set_id]['recovered_config']
                assert set(recovered_results) == set(own_config['test_scenarios'])
                assert all(set(rows) <= set(range(1, TARGET_RUNS + 1))
                           for rows in recovered_results.values())
                full_result_bindings[set_id] = recovered_results
            print(set_id, 'original full result contract recovered from verified archives; no histories fabricated')
        except saved_artifact_errors as error:
            blocked_restore.append(set_id)
            print(f"{set_id}: saved results rejected: {type(error).__name__}: {error}", flush=True)
            continue
else:
    print('Full-payload recovery not requested: completed JSON display remains read-only')


## Direct native missing-work API — opt-in only, never rerun completed work

The current 24 bindings cover the entire target, so this branch does nothing. Do not enable training simply to reconstruct completed scientific configuration. If work genuinely becomes missing, populate `original_native_records` from that subset's pinned original constructor/model records, not another pipeline or current factory defaults. Each record must contain the original `configuration` (including allocator class/kwargs) and `modelSettings`. Missing records cause a precise source-gap error only for genuinely missing work.

Every configuration call uses its own existing private `0700` temporary `staging_dir`. Evaluator construction has backups disabled; runner/model lookup is enabled before the direct native test call. No subprocess, wrapper, subclass or execution helper.


In [ ]:
RUN_MISSING_SCIENCE = False
display_saved_pipeline_results()
sys.path.insert(0, str(FRAMEWORK_ROOT / 'notebooks'))
from cohort_run_publication import SharedCohortStore
from p155_remote_archive import ExperimentArchive
from daqr.config.local_backup_manager import TeeStream

class GitExperimentArchive(ExperimentArchive):
    log_stream = None

    def begin(self, position):
        super().begin(position)
        try:
            self.log_stream = tempfile.NamedTemporaryFile(mode='w', encoding='utf-8',
                dir=self.root, prefix=f'experiment-{position}-', suffix='.log', delete=False)
            self.original_streams = sys.stdout, sys.stderr
            sys.stdout = TeeStream(sys.stdout, self.log_stream)
            sys.stderr = TeeStream(sys.stderr, self.log_stream)
        except OSError as error:
            print(f'Log capture pending for {self.key(position)}: {error}', flush=True)

    def close_log(self):
        if self.log_stream is not None:
            sys.stdout, sys.stderr = self.original_streams
            stream, self.log_stream = self.log_stream, None
            stream.close()

    def _handoff(self, key, checkpoint, models, evaluator, *, log=None):
        if self.log_stream is not None:
            log = Path(self.log_stream.name)
            report_optional('Log cleanup pending', self.close_log)
        super()._handoff(key, checkpoint, models, evaluator, log=log)
        if log is not None:
            self.log_backup_pending = not self.remove_verified_log(key, log)

    def finish(self, position, results, evaluator_checkpoint):
        try:
            super().finish(position, results, evaluator_checkpoint)
        except Exception as error:
            self.backup_pending = True
            print(f'Backup pending for {self.key(position)}: {error}', flush=True)
        finally:
            report_optional('Log cleanup pending', self.close_log)

cohort_runs = cohort_store.prepare_launch(pipeline_git_key,
    pipeline.runner_execution_id or pipeline_git_key, workflow_document['plan']['cohorts'])
print(f'Pipeline {PIPELINE_ID}: {len(cohort_runs)} Git cohort branches ready')

def read_saved_experiment_log(cohort, allocator, threat, position):
    run = cohort_runs[cohort]
    if not run.prior or not run.prior.get('tag'):
        print(f'{cohort}: no published Git checkpoint version yet.')
        return None
    pinned = cohort_store.open_published(**run.identity)
    data = pinned.read(f'{allocator}/{threat}/{position}', 'log')
    return data.decode('utf-8') if data is not None else None

git_status_reporter = None
native_staging_dirs = globals().get('native_staging_dirs', {})
original_native_records = globals().get('original_native_records', {})
if missing_work and not RUN_MISSING_SCIENCE:
    print('Missing work preserved; scientific execution not requested:', missing_work)
elif missing_work:
    from daqr.config.experiment_config import ExperimentConfiguration
    from daqr.core import qubit_allocator
    from daqr.evaluation.multi_run_evaluator import MultiRunEvaluator
    from daqr.config import tae_workflow
    try:
        for set_id, threat in dict.fromkeys((set_id, threat) for set_id, threat, _ in missing_work):
            runtime_config = bindings[set_id]['recovered_config']
            runtime_scope = SimpleNamespace(pipeline_storage=pipeline.storage_binding,
                allocator=runtime_config['allocator'], scale=runtime_config['scale'],
                base_capacity=runtime_config['base_capacity'])
            if git_status_reporter is None:
                sys.path.insert(0, str(FRAMEWORK_ROOT / 'notebooks'))
                from tae_background_status import BackgroundWorkflowStatus
                git_status_reporter = BackgroundWorkflowStatus.for_pipeline(tae_workflow.WorkflowStore(), 154)
            report_optional("Workflow status pending", git_status_reporter.start_scope, runtime_scope)
            report_optional("Monitoring pending", monitor.set_runtime, runtime_scope, phase='restore')
            try:
                if (set_id, threat) not in original_native_records:
                    raise ValueError('Exact own allocator/model constructor records required: ' + set_id + '/' + threat)
                original = original_native_records[(set_id, threat)]
                settings = deepcopy(original['configuration'])
                allocator_spec = settings.pop('allocator')
                model_settings = original['modelSettings']
                own_config = bindings[set_id]['recovered_config']
                assert settings['runs'] == TARGET_RUNS
                assert settings['models'] == own_config['models']
                settings.update(runs=TARGET_RUNS, overwrite=False, resume=True, use_last_backup=False,
                                scenarios={threat: own_config['test_scenarios'][threat]}, attack_type=threat)
                staging_dir = Path(tempfile.mkdtemp(prefix=f'quantum-pipeline-{PIPELINE_ID}-native-')).resolve()
                native_staging_dirs[(set_id, threat)] = staging_dir
                assert staging_dir.is_dir() and staging_dir.stat().st_mode & 0o077 == 0
                config = ExperimentConfiguration(
                    allocator=getattr(qubit_allocator, allocator_spec['class'])(**allocator_spec['kwargs']),
                    pipeline_storage=pipeline.storage_binding, staging_dir=staging_dir,
                    drive_service=pipeline_drive_service(), **settings)
                for model, recorded_model in model_settings.items():
                    assert model in config.algorithm_configs
                    for field in ('kwargs', 'seed_offset', 'runner_type'):
                        config.algorithm_configs[model][field] = deepcopy(recorded_model[field])
            except saved_artifact_errors as error:
                blocked_restore.append((set_id, threat))
                print(f"{set_id}/{threat}: saved restore rejected: {type(error).__name__}: {error}", flush=True)
                continue
            config.resume_scope = 'all'
            config.resume_allow_drive_downloads = False
            report_optional("Monitoring pending", monitor.set_runtime, config, phase='resume')
            evaluator = MultiRunEvaluator(configs=config, base_frames=own_config['base_frames'],
                frame_step=own_config['frame_step'], base_seed=own_config['base_seed'])
            cohort = 'S' + format(float(config.scale), 'g') + ('Tb' if config.base_capacity else 'T')
            cache = staging_dir / 'github-checkpoints'
            cache.mkdir(exist_ok=True)
            config.use_last_backup = False
            archive = GitExperimentArchive(config, cohort_runs[cohort], cache,
                drive_factory=GoogleDriveBackupManager.pipeline_creation_drive_service)
            archive.backup_pending = False
            pinned = cohort_store.open_published(**archive.run.identity) if archive.run.prior and archive.run.prior.get('tag') else None
            evaluator.env_experiments = {threat: {}}
            from daqr.evaluation.experiment_runner import QuantumExperimentRunner
            from evaluator_derived_results import read_completed_result
            saved_rows = evaluator.env_experiments[threat]
            for position in range(1, TARGET_RUNS + 1):
                if position in saved_rows or pinned is None or archive.key(position) not in pinned.experiments:
                    continue
                try:
                    frames = own_config['base_frames'] + (position - 1)*own_config['frame_step']
                    entry = pinned.experiments[archive.key(position)]
                    if not entry.get('state') and entry.get('recoveredState'):
                        saved_rows[position] = read_completed_result(pinned, archive.key(position),
                            pipeline=f'Pipeline_{PIPELINE_ID}', set_id=set_id, threat=threat,
                            position=position, frames=frames, models=config.models)
                        continue
                    config.backup_mgr.github_checkpoint_search = pinned.checkpoint_search(archive.key(position), cache)
                    capacity = own_config['base_frames'] if config.base_capacity else frames
                    runner = QuantumExperimentRunner(id=position, config=config, frames_count=frames,
                        capacity=capacity, base_seed=own_config['base_seed'] + (position - 1)*100,
                        attack_type=threat, attack_intensity=config.attack_intensity)
                    config.use_last_backup = True
                    config.resume_obj(runner)
                    if (runner.resumed and runner.is_complete and runner.id == position
                            and runner.frames_count == frames and set(runner.results) == set(config.models)):
                        saved_rows[position] = dict(results=deepcopy(runner.results), winner=runner.winner, exp_id=position)
                except (OSError, ValueError, KeyError, RuntimeError, pickle.UnpicklingError) as error:
                    print(f'{set_id}/{threat}/{position}: Git resume pending: {error}', flush=True)
                finally:
                    config.use_last_backup = False
            if any(archive.key(position) in archive.run.experiments and position not in saved_rows
                   for position in range(1, TARGET_RUNS + 1)):
                blocked_restore.append((set_id, threat))
                print(f'{set_id}/{threat}: unresolved published checkpoint; not rerunning.', flush=True)
                evaluator.cleanup(verbose=False, cooldown_seconds=0)
                continue
            workflow_rows = (workflow_document.get('allocators', {}).get(set_id.split(':', 1)[0], {})
                .get('cohorts', {}).get(cohort, {}).get('threats', {}).get(threat, {}).get('experiments', {}))
            completed_positions = {int(position) for position, item in workflow_rows.items()
                                   if item.get('state') == 'done'}
            completed_positions.update(int(position) for position in bindings[set_id]['env_experiments'][threat])
            unresolved = completed_positions - saved_rows.keys()
            if unresolved:
                for position in sorted(unresolved):
                    print(f'{set_id}/{threat}/{position}: completed result history unavailable; skipped, no rerun.')
                blocked_restore.append((set_id, threat))
                evaluator.cleanup(verbose=False, cooldown_seconds=0)
                continue
            config.remote_experiment_archive = True
            evaluator._experiment_archive = archive
            try:
                report_optional("Monitoring pending", monitor.set_runtime, config, phase='experiments')
                evaluator.test_stochastic_environment(attack_type=threat,
                    scenarios={threat: own_config['test_scenarios'][threat]}, cal_winner=True, parellel=False)
                evaluator.calculate_scenarios_performance()
                report_optional("Monitoring pending", monitor.set_runtime, config, phase='fairness')
                evaluator.calculate_fairness()
                report_optional("Monitoring pending", monitor.set_runtime, config, phase='cohort-done')
                report_optional("Workflow status pending", git_status_reporter.confirm_saved_cohort, monitor.document,
                    threats=monitor_plan['threats'], runs=monitor_plan['runs'], models=monitor_plan['models'])
                tae_workflow.sync_completed_status(config)
                report_optional("Monitoring pending", lambda: tae_workflow.current_tracker(config).write_live_status())
            finally:
                report_optional('Log cleanup pending', archive.close_log)
                evaluator.cleanup(verbose=False, cooldown_seconds=0)
    except BaseException:
        if getattr(monitor, 'runtime', None) is not None:
            report_optional("Monitoring pending", monitor.set_runtime, runtime_scope, phase='pipeline-error', state='stopped')
            if git_status_reporter is not None:
                report_optional("Workflow status pending", git_status_reporter.finish_run, success=False)
        raise
    else:
        if git_status_reporter is not None:
            report_optional("Workflow status pending", git_status_reporter.finish_run, success=not blocked_restore)
        report_optional("Monitoring pending", monitor.set_runtime, runtime_scope, phase='restore-pending' if blocked_restore else 'pipeline-complete',
                        state='stopped' if blocked_restore else 'done')

else:
    print('All original target positions are complete: zero native scientific calls')


## Read-only fairness/preparation reuse — verify persisted outputs, do not rebuild

Each own workflow already has fairness and prepared dataset references. Verify every referenced fairness artifact and the whole prepared dataset by size/SHA256 before reporting reuse. Any mismatch stops without overwriting evidence. No helper execution object, migration, storage creation, or preparation regeneration. Reward/frame-horizon fairness proxies are not measured service equity or network latency.


In [ ]:
fairness_verified = {}
for set_id in bindings:
    try:
        references = workflow_document.get('savedFairness', {}).get(set_id)
        if references is None or set(references) != {'manifest', 'outcomes', 'records', 'summary'}:
            raise ValueError('Missing original fairness inputs: ' + set_id)
        for name, reference in references.items():
            key = (reference['provider_id'], reference['content_sha256'], reference['byte_length'])
            if key not in saved_byte_cache:
                saved_byte_cache[key] = pipeline_drive_service().files().get_media(
                    fileId=reference['provider_id'], supportsAllDrives=True).execute()
            raw = saved_byte_cache[key]
            assert len(raw) == reference['byte_length']
            assert hashlib.sha256(raw).hexdigest() == reference['content_sha256'], (set_id, name)
            json.loads(raw)
        fairness_verified[set_id] = deepcopy(references)
    except saved_artifact_errors as error:
        print(f"{set_id}: saved fairness verification pending: {type(error).__name__}: {error}", flush=True)
        continue
try:
    reference = workflow_document.get('preparedDataset')
    if reference is None:
        raise ValueError('Own prepared dataset reference missing; preserve outputs, no rebuild authorized')
    key = (reference['provider_id'], reference['content_sha256'], reference['byte_length'])
    if key not in saved_byte_cache:
        saved_byte_cache[key] = pipeline_drive_service().files().get_media(
            fileId=reference['provider_id'], supportsAllDrives=True).execute()
    raw = saved_byte_cache[key]
    assert len(raw) == reference['byte_length']
    assert hashlib.sha256(raw).hexdigest() == reference['content_sha256']
    prepared_dataset = json.loads(raw)
    assert prepared_dataset['coverage']['complete']
    assert prepared_dataset['coverage']['expectedMembers'] == 24
    assert prepared_dataset['coverage']['observedMembers'] == 24
    assert prepared_dataset['coverage']['expectedOutcomes'] == 24*5*TARGET_RUNS*5
    assert prepared_dataset['coverage']['observedOutcomes'] == 24*5*TARGET_RUNS*5
    assert prepared_dataset['publicationReadiness']['ready']
    print('Verified original fairness subsets:', len(fairness_verified))
    print('Verified original prepared coverage:', prepared_dataset['coverage'])
    print('Reused saved outputs; no scientific execution, fairness recomputation or preparation rebuild')
except saved_artifact_errors as error:
    prepared_dataset = None
    print(f"Saved preparation verification pending: {type(error).__name__}: {error}", flush=True)
